# V2 Dataset GenerationGenerate a synthetic training dataset using:- **Normalized colour palette** (`categorized_colors_normalized.csv`)- **7 pattern types** (v1's 4 + plaid, gradient, chevron)- **V2 augmentation pipeline** (v1 + perspective warp, specular highlight, vignette)- **Overlay blend mode** for fold textures (in addition to multiply)Output: `datasets/generated_v2/images/` + `datasets/generated_v2/labels.csv`

In [ ]:
import sys, os, importlibimport numpy as npimport pandas as pdimport cv2import matplotlib.pyplot as pltfrom tqdm.auto import tqdmPROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))if PROJECT_ROOT not in sys.path:    sys.path.insert(0, PROJECT_ROOT)import utils.color_utils_extended as cueimportlib.reload(cue)# ── Configuration ────────────────────────────────────────────────────────────COLOR_CLASSES = [    "red", "orange", "yellow", "green", "blue",    "violet", "purple", "white", "gray", "black",    "pink", "brown", "olive",]NUM_CLASSES = len(COLOR_CLASSES)N_TRAIN = 20000N_VAL   = 2000OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'datasets', 'generated_v2')IMAGE_DIR  = os.path.join(OUTPUT_DIR, 'images')CSV_PATH   = os.path.join(OUTPUT_DIR, 'labels.csv')BG_DIR     = os.path.join(PROJECT_ROOT, 'datasets', 'indoorCVPR_09_modified')NORMALIZED_CSV = os.path.join(PROJECT_ROOT, 'datasets', 'categorized_colors_normalized.csv')if not os.path.exists(NORMALIZED_CSV):    raise FileNotFoundError(        f"Run dataset_preparation.ipynb first to create:\n  {NORMALIZED_CSV}"    )gen = cue.DatasetGeneratorV2(    csv_path=NORMALIZED_CSV,    path_to_bgs=BG_DIR,)print(f"Generator ready.  Patterns: {cue.PATTERN_TYPES_V2}")print(f"Train: {N_TRAIN}  |  Val: {N_VAL}  |  Output: {OUTPUT_DIR}")

## Preview SamplesGenerate a small grid to visually inspect the new patterns and augmentations.

In [ ]:
n_rows, n_cols = 5, 6fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 15))for r in range(n_rows):    for c in range(n_cols):        img, labels = gen.generate()        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        ax = axes[r][c]        ax.imshow(img_rgb)        top_label = max(labels, key=labels.get)        top_pct = labels[top_label]        ax.set_title(f"{top_label} ({top_pct:.0f}%)", fontsize=9)        ax.axis('off')plt.suptitle('V2 Dataset Preview (30 random samples)', fontsize=14, y=1.01)plt.tight_layout()plt.show()

## Full Generation LoopGenerates the complete train + val dataset with resume support.

In [ ]:
os.makedirs(IMAGE_DIR, exist_ok=True)# Resume support: check existing labelsexisting = set()if os.path.exists(CSV_PATH):    existing_df = pd.read_csv(CSV_PATH)    existing = set(existing_df['filename'].tolist())    print(f"Resuming: {len(existing)} images already generated")csv_mode = 'a' if existing else 'w'csv_file = open(CSV_PATH, csv_mode, newline='')if not existing:    header = 'filename,split,' + ','.join(COLOR_CLASSES) + '\n'    csv_file.write(header)total = N_TRAIN + N_VALpbar = tqdm(total=total - len(existing), desc="Generating")for i in range(total):    split = 'train' if i < N_TRAIN else 'val'    filename = f"{split}_{i:06d}.jpg"    if filename in existing:        continue    img, label_pcts = gen.generate()    # Normalize to sum to 1.0    label_vec = [label_pcts.get(c, 0.0) for c in COLOR_CLASSES]    total_pct = sum(label_vec)    if total_pct > 0:        label_vec = [v / total_pct for v in label_vec]    else:        label_vec = [1.0 / NUM_CLASSES] * NUM_CLASSES    # Save image    img_path = os.path.join(IMAGE_DIR, filename)    cv2.imwrite(img_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])    # Write CSV row    csv_file.write(filename + ',' + split + ',' + ','.join(f'{v:.6f}' for v in label_vec) + '\n')    csv_file.flush()    pbar.update(1)pbar.close()csv_file.close()print("Generation complete.")

## Quality EvaluationVerify dataset integrity and inspect label distributions.

In [ ]:
labels_df = pd.read_csv(CSV_PATH)print(f"Total samples: {len(labels_df)}")print(f"  Train: {(labels_df['split'] == 'train').sum()}")print(f"  Val:   {(labels_df['split'] == 'val').sum()}")# Check label sumslabel_sums = labels_df[COLOR_CLASSES].sum(axis=1)print(f"\nLabel sum — min: {label_sums.min():.4f}, max: {label_sums.max():.4f}, "      f"mean: {label_sums.mean():.4f}")# Mean label distribution (how often each colour appears as dominant)mean_dist = labels_df[COLOR_CLASSES].mean()dominant_counts = labels_df[COLOR_CLASSES].idxmax(axis=1).value_counts()fig, axes = plt.subplots(1, 2, figsize=(16, 5))# Mean probability per classaxes[0].bar(COLOR_CLASSES, mean_dist.values, color='steelblue', edgecolor='white')axes[0].set_ylabel('Mean probability')axes[0].set_title('Average Label Distribution Across Dataset')axes[0].set_xticklabels(COLOR_CLASSES, rotation=45, ha='right')axes[0].axhline(1.0 / NUM_CLASSES, color='red', ls='--', alpha=0.5, label='Uniform')axes[0].legend()# Dominant class distributiondom_sorted = dominant_counts.reindex(COLOR_CLASSES, fill_value=0)axes[1].bar(COLOR_CLASSES, dom_sorted.values, color='darkorange', edgecolor='white')axes[1].set_ylabel('Count (dominant class)')axes[1].set_title('Dominant Colour Distribution')axes[1].set_xticklabels(COLOR_CLASSES, rotation=45, ha='right')plt.tight_layout()plt.show()# Show a few random samples with labelsfig, axes = plt.subplots(2, 5, figsize=(18, 7))samples = labels_df.sample(10)for ax, (_, row) in zip(axes.flat, samples.iterrows()):    img = cv2.imread(os.path.join(IMAGE_DIR, row['filename']))    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    ax.imshow(img)    top3 = sorted([(c, row[c]) for c in COLOR_CLASSES], key=lambda x: -x[1])[:3]    title = '\n'.join(f'{c}: {p:.1%}' for c, p in top3)    ax.set_title(title, fontsize=8)    ax.axis('off')plt.suptitle('Random Samples with Top-3 Labels', fontsize=13)plt.tight_layout()plt.show()

## Create ZIP for Colab UploadCreates a zip file ready to upload to the shared Google Drive folder.

In [ ]:
import zipfileZIP_PATH = os.path.join(PROJECT_ROOT, 'datasets', 'generated_v2.zip')print(f"Creating zip: {ZIP_PATH}")with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:    # Add labels.csv    zf.write(CSV_PATH, os.path.join('generated_v2', 'labels.csv'))    # Add all images    for fname in tqdm(os.listdir(IMAGE_DIR), desc="Zipping"):        fpath = os.path.join(IMAGE_DIR, fname)        zf.write(fpath, os.path.join('generated_v2', 'images', fname))zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)print(f"Done: {zip_size_mb:.1f} MB")print(f"\nUpload this zip to the shared Google Drive folder, then run training.ipynb on Colab.")